# 04 | Cross-validation and data diagnostics

Holdout splitting, cross-validation, user/name checks, activity analysis, and class imbalance.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

train = load_events("train")
dataset_model = pd.read_parquet(data_path("dataset_model.parquet"))


# Train, validation, and test splits

In [20]:
rng = np.random.default_rng(42)

all_users = dataset_model["userId"].unique()
shuffled_users = rng.permutation(all_users)

n_holdout = int(0.20 * len(shuffled_users))

holdout_users = set(shuffled_users[:n_holdout])
dev_users = shuffled_users[n_holdout:]

final_cutoff = CUTOFFS[-1] ## final test

dev_data = dataset_model.loc[
    (dataset_model["cutoff"] < final_cutoff) &
    (~dataset_model["userId"].isin(holdout_users))
].reset_index(drop=True)

holdout_data = dataset_model.loc[
    (dataset_model["cutoff"] == final_cutoff) &
    (dataset_model["userId"].isin(holdout_users))
].reset_index(drop=True)

##### Construct two chronological cross-validation folds

# Two distinct groups of validation users
shuffled_dev = rng.permutation(dev_users)
n_val = int(0.25 * len(shuffled_dev))

val_groups = [
    set(shuffled_dev[:n_val]),
    set(shuffled_dev[n_val:2*n_val])
]

cv_splits = []

for i, val_cutoff in enumerate(CUTOFFS[1:-1]):

    val_users = val_groups[i]

    train_mask = (
        (dev_data["cutoff"] < val_cutoff) &
        (~dev_data["userId"].isin(val_users))
    )

    valid_mask = (
        (dev_data["cutoff"] == val_cutoff) &
        (dev_data["userId"].isin(val_users))
    )

    train_idx = np.flatnonzero(train_mask)
    valid_idx = np.flatnonzero(valid_mask)

    cv_splits.append((train_idx, valid_idx))

    train_users = set(dev_data.iloc[train_idx]["userId"])
    valid_users = set(dev_data.iloc[valid_idx]["userId"])

    # Checks to prevent data leakage
    assert train_users.isdisjoint(valid_users)

    assert (
        dev_data.iloc[train_idx]["cutoff"].max() + HORIZON
        <= val_cutoff
    )

    print(f"\nFold {i+1}")
    print("Validation cutoff:", val_cutoff.date())
    print("Train :", len(train_idx), "observations")
    print("Validation :", len(valid_idx), "observations")
    print("Users in both sets:", len(train_users & valid_users))


### Final holdout verification
assert set(dev_data["userId"]).isdisjoint(
    set(holdout_data["userId"])
)

print("\nHoldout final :", len(holdout_data), "observations")
print("Date :", final_cutoff.date())
print("Number of folds:", len(cv_splits))



Fold 1
Validation cutoff: 2018-10-21
Train : 8491 observations
Validation : 3132 observations
Users in both sets: 0

Fold 2
Validation cutoff: 2018-10-31
Train : 17832 observations
Validation : 3089 observations
Users in both sets: 0

Holdout final : 3063 observations
Date : 2018-11-10
Number of folds: 2


In [22]:

# ============================================
# 1. UNKNOWN USER IDENTIFIERS
# ============================================

unknown_values = [
    "unknown", "anonymous", "none",
    "nan", "null", "", "0", "guest"
]

print("Missing user IDs:")
print(train["userId"].isna().sum())

unknown_users = train[
    train["userId"].astype("string")
    .str.strip().str.lower()
    .isin(unknown_values)
]

print("\nGeneric user IDs:")
print(unknown_users["userId"].value_counts(dropna=False))


# ============================================
# 2. FIRST AND LAST NAMES
# ============================================

# One row per user ID and name combination
users_names = train[
    ["userId", "firstName", "lastName"]
].drop_duplicates().copy()

# Standardize the spelling and casing of names
for col in ["firstName", "lastName"]:
    users_names[col] = (
        users_names[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Identify missing or generic names
generic_names = {
    "unknown", "anonymous", "none",
    "nan", "null", "", "guest", "user"
}

suspicious_names = users_names[
    users_names["firstName"].isna() |
    users_names["lastName"].isna() |
    users_names["firstName"].isin(generic_names) |
    users_names["lastName"].isin(generic_names)
]

print("\nAccounts with suspicious names:")
print(suspicious_names["userId"].nunique())

display(suspicious_names.head(20))


# ============================================
# 3. MULTIPLE USER IDS SHARING THE SAME NAME
# ============================================

names_counts = (
    users_names
    .groupby(["firstName", "lastName"], dropna=False)
    .agg(
        n_userIds=("userId", "nunique"),
        exemples_ids=("userId",
                      lambda x: list(x.unique()[:5]))
    )
    .sort_values("n_userIds", ascending=False)
)

print("\nNames associated with multiple user IDs:")

display(
    names_counts[names_counts["n_userIds"] > 1]
    .head(20)
)

# Check the reverse: multiple names for one user ID
names_per_id = users_names.groupby("userId").size()

print("\nUser IDs associated with multiple names:")
print((names_per_id > 1).sum())


# ============================================
# 4. USERS WITH UNUSUALLY HIGH ACTIVITY
# ============================================

user_activity = (
    train.groupby("userId")
    .size()
    .sort_values(ascending=False)
)

print("\nTop 10 most active users:")
display(user_activity.head(10))

print("\nDistribution of events per user:")
display(
    user_activity.describe(
        percentiles=[0.5, 0.9, 0.99]
    )
)



Missing user IDs:
0

Generic user IDs:
Series([], Name: count, dtype: int64)

Accounts with suspicious names:
3


,userId,firstName,lastName
1188683,1857840,unknown,daniels
5978793,1096593,unknown,whitney
23005310,1181709,unknown,cannon



Names associated with multiple user IDs:


,,n_userIds,exemples_ids
firstName,lastName,,
isabella,jones,5,"[1102646, 1980447, 1230769, 1366075, 1754025]"
alexis,smith,4,"[1573034, 1461607, 1551778, 1397940]"
jaden,smith,4,"[1982471, 1811170, 1322329, 1928139]"
elijah,smith,4,"[1103800, 1920952, 1204451, 1448599]"
liam,johnson,4,"[1273639, 1751978, 1838545, 1301218]"
isabella,smith,4,"[1619792, 1411896, 1476032, 1353786]"
landon,smith,4,"[1384966, 1768591, 1235019, 1671328]"
logan,hill,4,"[1704999, 1005357, 1281072, 1130898]"
matthew,jones,4,"[1697183, 1643820, 1376842, 1971769]"



User IDs associated with multiple names:
0

Top 10 most active users:


userId
1931933    10998
1564221    10957
1006695     9919
1336969     9693
1730425     9252
1560520     9092
1662781     9006
1113757     8981
1989862     8929
1608306     8738
dtype: int64


Distribution of events per user:


count    19140.000000
mean       914.296552
std       1079.652218
min          1.000000
50%        537.500000
90%       2245.000000
99%       5131.220000
max      10998.000000
dtype: float64

In [23]:

# ============================================
# COMPARE ACTIVITY BY CHURN STATUS
# ============================================

activity_cols = [
    "n_sessions_10d",
    "n_active_days_10d",
    "n_songs_10d"
]

# 1. Mean and median by target class
activity_summary = (
    dataset_model
    .groupby("churn")[activity_cols]
    .agg(["mean", "median"])
    .round(2)
)

print("Mean and median activity by churn status:")
display(activity_summary)


# 2. Compare users within each prediction window
activity = dataset_model[
    ["userId", "cutoff", "churn"] + activity_cols
].copy()

# User activity percentile based on song events
activity["activity_percentile"] = (
    activity.groupby("cutoff")["n_songs_10d"]
    .rank(pct=True)
)

# Group users by their activity level
activity["activity_group"] = pd.cut(
    activity["activity_percentile"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[
        "Very low",
        "Low",
        "Medium",
        "High",
        "Very high"
    ],
    include_lowest=True
)

# 3. Churn rate by activity level
churn_by_activity = (
    activity.groupby("activity_group", observed=True)
    .agg(
        n_observations=("churn", "size"),
        n_churn=("churn", "sum"),
        churn_rate=("churn", "mean")
    )
)

churn_by_activity["churn_rate"] *= 100

print("\nChurn rate by activity level:")
display(churn_by_activity.round(2))



Mean and median activity by churn status:


n_sessions_10d        n_active_days_10d        n_songs_10d       
                mean median              mean median        mean median
churn                                                                  
0               2.66    2.0              2.59    2.0      174.39   80.0
1               4.60    4.0              4.11    4.0      328.29  216.0


Churn rate by activity level:


,n_observations,n_churn,churn_rate
activity_group,,,
Very low,12537,292,2.33
Low,11611,340,2.93
Medium,12117,510,4.21
High,12071,738,6.11
Very high,12098,1210,10.00


In [24]:

# ============================================
# CLASS IMBALANCE ANALYSIS
# ============================================

# 1. Overall class distribution
print("Overall distribution:")
print(dataset_model["churn"].value_counts())

print("\nPercentages:")
print(
    dataset_model["churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# 2. Class ratio
n_negative = (dataset_model["churn"] == 0).sum()
n_positive = (dataset_model["churn"] == 1).sum()

print(
    "\nNon-churn to churn ratio:",
    round(n_negative / n_positive, 2)
)

# 3. Class imbalance in each fold
for i, (train_idx, valid_idx) in enumerate(cv_splits, 1):

    y_train_fold = dev_data.iloc[train_idx]["churn"]
    y_valid_fold = dev_data.iloc[valid_idx]["churn"]

    print(f"\nFold {i}")

    print("Train :")
    print(y_train_fold.value_counts())

    print("Training churn rate:",
          round(y_train_fold.mean() * 100, 2), "%")

    print("Validation :")
    print(y_valid_fold.value_counts())

    print("Validation churn rate:",
          round(y_valid_fold.mean() * 100, 2), "%")

# 4. Final holdout
print("\nHoldout final :")
print(holdout_data["churn"].value_counts())

print("Holdout churn rate:",
      round(holdout_data["churn"].mean() * 100, 2), "%")


Overall distribution:
churn
0    57344
1     3090
Name: count, dtype: int64

Percentages:
churn
0    94.89
1     5.11
Name: proportion, dtype: float64

Non-churn to churn ratio: 18.56

Fold 1
Train :
churn
0    7979
1     512
Name: count, dtype: int64
Training churn rate: 6.03 %
Validation :
churn
0    2981
1     151
Name: count, dtype: int64
Validation churn rate: 4.82 %

Fold 2
Train :
churn
0    16865
1      967
Name: count, dtype: int64
Training churn rate: 5.42 %
Validation :
churn
0    2908
1     181
Name: count, dtype: int64
Validation churn rate: 5.86 %

Holdout final :
churn
0    2924
1     139
Name: count, dtype: int64
Holdout churn rate: 4.54 %


In [ ]:
# Preserve the exact rows and indices defining the folds
save_cv_state(dev_data, holdout_data, cv_splits)
print("Cross-validation folds and holdout saved for model training.")